# UltraGliClass: one resumable training session (stage 1 by default)

GLiClass uni-encoder on `dmis-lab/UltraBERT-150M`. **One run of this notebook = one Kaggle session.**
It restores the previous session's checkpoints, trains until the time budget, saves, and stops. Re-run it
(with the previous version's output attached as an input) until `STATUS.json` says `complete`.

**Inputs to attach** (Add data):
1. the code, as a private Kaggle Dataset (e.g. `ultragliclass-code`), and
2. from session 2 on, the *output of the previous version of this notebook* (holds `runs/` and `data/`).

**Settings:** Accelerator *GPU T4 x2*, Internet *on* (pip + Hugging Face downloads), Persistence off.


In [ ]:
# ---- knobs (S0A, per STAGE1_JOB_SPEC.md) ----
import glob as _g, os as _o
def _find(root, rel, up):
    hits = sorted(_g.glob(root + "/**/" + rel, recursive=True))
    print("search", rel, "->", hits)
    if not hits: return None
    p = hits[0]
    for _ in range(up): p = _o.path.dirname(p)
    return p
STAGE_CONFIG  = "configs/stage0_kaggle.yaml"
DATA_RECIPE   = "configs/data_stage0_kaggle.yaml"
DATA_DIR      = "data/stage0k"
PREP_ARGS     = "--streaming"
_codes = sorted(_o.path.dirname(h) for h in _g.glob("/kaggle/input/**/train.py", recursive=True))
_codes = sorted(_codes, key=lambda p: (0 if "ultragliclass-code" in p else 1, len(p)))
CODE_INPUT    = _codes[0] if _codes else "/kaggle/input/ultragliclass-code"
PREV_OUTPUT   = None
SESSION_HOURS = 12.0
RESERVE_HOURS = 0.75
QUICK_EVAL    = False
EXTRA_SET     = ["train.time_budget_hours=0.07"]
print("CODE_INPUT", CODE_INPUT, "PREV_OUTPUT", PREV_OUTPUT)
import time; T0 = time.time()

In [ ]:
import os, shutil, subprocess, sys, glob, json
WORK = "/kaggle/working/ultragliclass"
if not os.path.exists(os.path.join(WORK, "train.py")):
    src = CODE_INPUT if os.path.exists(os.path.join(CODE_INPUT, "train.py")) else glob.glob(CODE_INPUT + "/**/train.py", recursive=True)[0].rsplit("/", 1)[0]
    shutil.copytree(src, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns("runs", "data", "smoke_logs", "results"))
os.chdir(WORK)
# keep Kaggle's CUDA torch: pin it as a constraint so pip can never replace it
torch_pin = subprocess.run("pip freeze | grep -E '^torch=='", shell=True, capture_output=True, text=True).stdout.strip()
open("/tmp/constraints.txt", "w").write(torch_pin + "\n")
print("torch pin:", torch_pin)
subprocess.run(f"pip install -q -c /tmp/constraints.txt -r requirements.txt", shell=True, check=True)
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)
import torch, transformers, gliclass
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count(),
      "bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported())
print("transformers", transformers.__version__, "gliclass", gliclass.__version__ if hasattr(gliclass, "__version__") else "?")

In [ ]:
# restore checkpoints + prepared data from the previous session's output (if attached)
if PREV_OUTPUT:
    prev = PREV_OUTPUT if os.path.exists(os.path.join(PREV_OUTPUT, "runs")) else glob.glob(PREV_OUTPUT + "/**/runs", recursive=True)[0].rsplit("/", 1)[0]
    for sub in ("runs", "data", "results"):
        if os.path.exists(os.path.join(prev, sub)):
            shutil.copytree(os.path.join(prev, sub), sub, dirs_exist_ok=True)
            print("restored", sub)
for st in glob.glob("runs/*/STATUS.json"):
    print(st, json.load(open(st)).get("state"), json.load(open(st)).get("global_step"))

In [ ]:
# prepare data once (deterministic: same seed -> same rows in every session), then reuse it
if not os.path.exists(os.path.join(DATA_DIR, "train.jsonl")):
    subprocess.run(f"python prepare_data.py --recipe {DATA_RECIPE} {PREP_ARGS}", shell=True, check=True)
print(open(os.path.join(DATA_DIR, "manifest.json")).read()[:1500])

In [ ]:
# train until done or out of time; train.py saves a checkpoint and exits cleanly at the budget
elapsed_h = (time.time() - T0) / 3600
budget = max(0.25, SESSION_HOURS - elapsed_h - RESERVE_HOURS)
ngpu = max(1, torch.cuda.device_count())
sets = " ".join([f"train.time_budget_hours={budget:.2f}"] + EXTRA_SET)
launcher = f"torchrun --nproc_per_node {ngpu}" if ngpu > 1 else "python"
cmd = f"{launcher} train.py --config {STAGE_CONFIG} --resume auto --set {sets}"
print(cmd, flush=True)
rc = subprocess.run(cmd, shell=True).returncode
print("exit code", rc)

In [ ]:
sys.path.insert(0, WORK)
from ugc.config import load_config
out_dir = load_config(STAGE_CONFIG, EXTRA_SET)["train"]["output_dir"]
status = json.load(open(os.path.join(out_dir, "STATUS.json")))
print(json.dumps(status, indent=2))
if status["state"] == "needs_resume":
    print("\nNOT DONE: save this version, then run the notebook again with PREV_OUTPUT = this version's output.")
elif status["state"] == "nan_stop":
    print("\nSTOPPED ON NaN/inf loss: try precision fp32 for the first steps, a lower LR, or check data.")

In [ ]:
# quick zero-shot eval once the stage is complete (full eval runs in its own session)
if QUICK_EVAL and status["state"] == "complete":
    subprocess.run(f"python eval.py --model {status['final_model']} --suites gliclass intent typed "
                   f"--max_per_dataset 500 --permutations 3 --calib_file {DATA_DIR}/val.jsonl "
                   f"--out results/{os.path.basename(out_dir)}_quick.json", shell=True)

In [ ]:
# keep the output small: drop the older checkpoint (the newest one is enough to resume)
ckpts = sorted(glob.glob(os.path.join(out_dir, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
for c in ckpts[:-1]:
    shutil.rmtree(c)
subprocess.run("du -sh runs data results 2>/dev/null", shell=True)
print("session hours used:", round((time.time() - T0) / 3600, 2))